(course-alzheimer-energy-minimization)=

# Path A - Module 43: Energy Minimization

Experimental structures and manually built models often contain "hot spots": atoms that are too close or bonds that are stretched. Such defects can cause numerical instability; energy minimization is one preparation step rather than a guarantee of simulation readiness.

In this module, you will learn to perform **Energy Minimization** to relax your complex and prepare it for molecular dynamics.

The fibril/peptide recipe assumes consistent chemical states, force-field parameters and a deliberate spatial assembly. Building and merging independent molecules does not establish a bound pose. Select one structure explicitly before energy evaluation.

For a native `MolSys`, the default call returns an independent system. The examples retain it in `molsys`, so later analysis uses the minimized coordinates. To edit a native input deliberately, use `in_place=True` and retain the existing variable; that mode returns `None`. The supported method is L-BFGS with engine default stopping settings; this public API has no `tolerance` argument. See {ref}`the minimization tutorial <Tutorial_Potential_energy_minimization>`.

An existing OpenMM Context/Simulation cannot currently be copied independently. Such requests, including explicit output forms, are rejected before optimization. Use the native independent workflow above, or explicitly choose in-place engine editing when that ownership is intended.


Before constructing a mechanical model, inspect the [effective model recipe](../../user/tools/molecular_mechanics/get_model_recipe.md). Stored unsupported tuning requests now refuse with named-field diagnostics; they are not silently applied. The recipe distinguishes physical controls from execution settings and does not prepare chemistry or change coordinates.


In [ ]:
import molsysmt as msm
from molsysmt import systems

# Load our solvated complex
fibril = msm.convert('pdb:2BEG', to_form='molsysmt.MolSys')
msm.build.add_missing_hydrogens(fibril)
peptide = msm.build.build_peptide('KLVFF')
msm.build.add_missing_hydrogens(peptide)
molsys = msm.merge([fibril, peptide])
molsys = msm.extract(molsys, structure_indices=0)

# Check energy before minimization
energy_before = msm.molecular_mechanics.get_potential_energy(molsys)
print(f"Potential Energy before relaxation: {energy_before}")

## 1. The Minimization Process
The implemented method is L-BFGS local optimization. It searches the prepared force-field model from the supplied coordinates using the engine's stopping and constraint settings. Individual forces need not vanish under constraints; a completed search does not establish the nearest or global minimum.


In [ ]:
# Perform energy minimization
# Note: This might take a few seconds depending on your hardware
molsys_before = molsys
molsys = msm.molecular_mechanics.potential_energy_minimization(molsys)

print("Minimization completed.")

## 2. Verifying Success
Compare finite energies before and after using the same physical model, constraints and units. A prepared model with removable strain may decrease in energy; the amount and sign are model-dependent, and a structure already satisfying the stopping criterion need not move appreciably. A lower energy does not establish binding or thermodynamic stability.


In [ ]:
energy_after = msm.molecular_mechanics.get_potential_energy(molsys)
print(f"Potential Energy after relaxation: {energy_after}")

diff = energy_before - energy_after
print(f"Energy reduction: {diff}")

## 3. Inspecting Structural Changes
Inspect the minimized coordinates and compare them with a deliberately retained original system if you need RMSD or geometry checks. Local optimization does not guarantee preservation of every structural feature or an equilibrated conformation.


In [ ]:
# Inspect the retained minimized system; the original was not changed in place.
msm.view(molsys)


---

## 🏆 Path A Challenge: The Relaxer

1. Prepare a model with a controlled geometric perturbation and supported chemistry/parameters.
2. Record its initial energy and geometry using explicit units.
3. Minimize it, retain the returned system and measure elapsed time; this API does not return an iteration count.
4. Check energies, bond geometry and coordinate changes instead of assuming that completion proves physical stability.

Minimization has optimized this prepared configuration locally. It has not established equilibrium or a stable bound complex. Continue with [Module 44: AMBER TLeap](44_AMBER_TLeap.ipynb) to examine force-field assignment; the optimization above already requires a valid parameterized model.


(course-alzheimer-energy-minimization-see-also)=
:::{seealso}
:class: dropdown

- {ref}`Minimization tutorial <Tutorial_Potential_energy_minimization>` — result ownership, units and supported options.
- {ref}`Potential energy <Tutorial_Get_potential_energy>` — compare a prepared model before and after optimization.
- [Module 44: AMBER TLeap](44_AMBER_TLeap.ipynb) — force-field assignment.
:::
